In [ ]:
# Проверка задания п2.2.5 на устройстве: сравнивает частоты, расчёт и итерацию на рабочей и на пониженной частоте ядра
# и записывает обмен в файл device-2-2-5.log.

import time
from datetime import datetime

import serial
from serial.tools import list_ports

VENDOR_ID = 0x2E8A
PRODUCT_ID = 0x000A

TASK = "2.2.5"
PROJECT = "221-command-time"
LOG_NAME = "device-2-2-5.log"
# Команда и сколько секунд слушать плату, прежде чем отправить следующую
STEPS = [
    ("clk_info", 1),
    ("main_time_reset", 2),
    ("main_time_exec", 1),
    ("calc_pi", 6),
    ("clk_sys_low", 1),
    ("clk_info", 1),
    ("uptime", 2),
    ("uptime", 1),
    ("main_time_reset", 2),
    ("main_time_exec", 1),
    ("calc_pi", 11),
    ("clk_sys_default", 1),
    ("clk_info", 1),
    ("calc_pi", 6),
]


def find_board():
    for port in list_ports.comports():
        if port.vid == VENDOR_ID and port.pid == PRODUCT_ID:
            return port
    return None


def talk(board):
    exchange = []
    with serial.Serial(board.device, timeout=0.2) as port:
        time.sleep(0.2)
        port.reset_input_buffer()
        started = time.monotonic()
        for command, listen_s in STEPS:
            port.write((command + "\n").encode("ascii"))
            exchange.append((time.monotonic() - started, "-->", command))
            deadline = time.monotonic() + listen_s
            while time.monotonic() < deadline:
                line = port.readline().decode("ascii", "replace").strip()
                if line:
                    exchange.append((time.monotonic() - started, "<--", line))
                    print(line, end="\r\n")
    return exchange


def write_log(board, exchange):
    with open(LOG_NAME, "w", encoding="utf-8") as log:
        log.write("задание: " + TASK + "\n")
        log.write("проект: " + PROJECT + "\n")
        log.write("устройство: %04x:%04x\n" % (board.vid, board.pid))
        log.write("серийный номер: " + str(board.serial_number) + "\n")
        log.write("порт: " + board.device + "\n")
        log.write("начало: " + datetime.now().isoformat(timespec="seconds") + "\n")
        for moment, direction, text in exchange:
            log.write("%8.3f %s %s\n" % (moment, direction, text))
        answers = len(exchange) - len(STEPS)
        log.write("итог: отправлено команд %d, принято строк %d\n" % (len(STEPS), answers))


board = find_board()

if board is None:
    print("Плата не найдена. Проверьте кабель и запишите на плату прошивку задания.", end="\r\n")
else:
    print("Плата на порту " + board.device + ", отправляю команды", end="\r\n")
    exchange = talk(board)
    write_log(board, exchange)
    print("Обмен записан в " + LOG_NAME, end="\r\n")


Плата на порту COM3, отправляю команды
clk_info
dbg read_line:179 got clk_info
signal     set_khz measured_khz
clk_ref      12000        12001
clk_sys     125000       125000
clk_peri    125000       125000
clk_usb      48000        48000
clk_adc      48000        48000
rosc             -         5404
main_time_reset
dbg read_line:179 got main_time_reset
max reset
main_time_exec
dbg read_line:179 got main_time_exec
iteration avg 2.62 us, max 410 us
calc_pi
dbg read_line:179 got calc_pi
pi: 3.14159165
time: 4730 ms
clk_sys_low
dbg read_line:179 got clk_sys_low
inf clk_sys_set:17 clk_sys 62500 kHz
clk_info
dbg read_line:179 got clk_info
signal     set_khz measured_khz
clk_ref      12000        12002
clk_sys      62500        62500
clk_peri     48000        48000
clk_usb      48000        48000
clk_adc      48000        48000
rosc             -         5406
uptime
dbg read_line:179 got uptime
uptime: 21935 ms
uptime
dbg read_line:179 got uptime
uptime: 23967 ms
main_time_reset
dbg read_li